# Data exploration

FNCE 2370 — Team 7 | Predicting Corporate Distress Risk

Inspect the supplied merged Parquet file, identifiers, outcome columns, historical coverage, and CHS feature availability.

Run from the repository root (or its `notebooks/` subdirectory). See `README.md` for execution instructions. Source-data assumptions remain provisional pending TA confirmation.

## Exploration scope

We use the instructor's premerged CHS–Global Characteristics Parquet file rather than recreating an undocumented upstream merge. Inspect metadata and selected columns to avoid loading the full 223-column panel into memory. `permno`, `permco`, dates, outcome flags and merge flags are diagnostics or identifiers, not candidate predictors. The full-period summaries are descriptive; subsequent feature screening and transformation fitting use only eligible development observations.

The supplementary `00_source_provenance_checks.ipynb` tests candidate upstream keys, event histories and the effect of choosing a single representative security per company-month. Those checks do not change the modeling sample.

In [ ]:
import pandas as pd
from pathlib import Path

print(Path.cwd())

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

file_path = project_root / 'Project_5_data_chs_main_global.parquet'

print("File exists:", file_path.exists())

In [ ]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(file_path)

print("File size (GB):", round(file_path.stat().st_size / 1e9, 3))
print("Number of rows:", parquet_file.metadata.num_rows)
print("Number of columns:", parquet_file.metadata.num_columns)
print("Number of row groups:", parquet_file.metadata.num_row_groups)

print("\nColumn names:")
for col in parquet_file.schema_arrow.names:
    print(col)

In [ ]:
import pandas as pd

batch = next(parquet_file.iter_batches(batch_size=10))
df_preview = batch.to_pandas()

pd.set_option("display.max_columns", None)

display(df_preview)

In [ ]:
display(df_preview.dtypes)

In [ ]:
# Firm and security identifiers
id_cols = [
    "permno",
    "permco",
    "gvkey",
    "iid",
    "id"
]

# Date variables
date_cols = [
    "date",
    "month_end",
    "first_failure_month",
    "first_exit_month",
    "first_ambiguous_exit_month",
    "global_factor_eom",
    "global_factor_date"
]

# Failure and risk-set indicators
target_cols = [
    "at_risk",
    "failure_current_month",
    "failure_next_month",
    "failure_t_plus_1",
    "failure",
    "failure_within_6m",
    "failure_within_12m",
    "failure_within_24m",
    "failure_within_36m"
]

# Sample and merge indicators
sample_cols = [
    "global_factor_matched",
    "obs_main",
    "exch_main",
    "common",
    "primary_sec",
    "excntry",
    "crsp_shrcd",
    "crsp_exchcd",
    "source_crsp"
]

# Baseline CHS predictors
chs_features = [
    "nimtaavg",
    "tlmta",
    "exretavg",
    "sigma",
    "rsize",
    "cashmta",
    "mb",
    "price_log"
]

# Additional variables for feature validation
auxiliary_cols = [
    "nimta",
    "nita",
    "tlta",
    "exret",
    "sigma_fill",
    "prc",
    "market_equity",
    "ret_adj",
    "sprtrn"
]

selected_cols = (
    id_cols
    + date_cols
    + target_cols
    + sample_cols
    + chs_features
    + auxiliary_cols
)

print("Selected columns:", len(selected_cols))

In [ ]:
df = pd.read_parquet(
    file_path,
    columns=selected_cols
)

print("Dataset shape:", df.shape)

print(
    "Memory usage (GB):",
    round(df.memory_usage(deep=True).sum() / 1e9, 3)
)

display(df.head())

In [ ]:
for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

print("Date conversion completed.")
display(df[date_cols].dtypes)

In [ ]:
print("First observation:", df["month_end"].min())
print("Last observation:", df["month_end"].max())

print("Unique securities:", df["permno"].nunique())
print("Unique CRSP companies:", df["permco"].nunique())
print("Unique Compustat companies:", df["gvkey"].nunique())

print("Unique months:", df["month_end"].nunique())

print(
    "Missing security identifiers:",
    df["permno"].isna().sum()
)

print(
    "Missing month-end dates:",
    df["month_end"].isna().sum()
)

In [ ]:
annual_coverage = (
    df.assign(year=df["month_end"].dt.year)
    .groupby("year")
    .agg(
        observations=("permno", "size"),
        unique_securities=("permno", "nunique"),
        unique_companies=("permco", "nunique"),
        current_month_failures=("failure_current_month", "sum"),
        global_matches=("global_factor_matched", "sum")
    )
)

pd.set_option("display.max_rows", 100)

display(annual_coverage)

## Identifier, target and feature diagnostics

Check uniqueness at both security-month and company-month levels before imposing sample restrictions. Compare the supplied failure indicators and their missingness instead of automatically shifting the target. Part I resolves their timing and risk-set interpretation; Part II uses explicit feature allowlists to exclude forward returns, future event dates and other outcome-derived fields.

In [ ]:
print(
    "Duplicate security-month observations:",
    df.duplicated(
        subset=["permno", "month_end"]
    ).sum()
)

print(
    "Duplicate company-month observations:",
    df.loc[df["permco"].notna()].duplicated(
        subset=["permco", "month_end"]
    ).sum()
)

print(
    "Fully duplicated rows across loaded columns:",
    df.duplicated().sum()
)

In [ ]:
for col in target_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

In [ ]:
candidate_targets = [
    "failure_current_month",
    "failure_next_month",
    "failure_t_plus_1",
    "failure"
]

target_comparison = pd.DataFrame(
    index=candidate_targets,
    columns=candidate_targets,
    dtype="int64"
)

for col1 in candidate_targets:
    for col2 in candidate_targets:

        identical = (
            df[col1].eq(df[col2]).fillna(False)
            | (df[col1].isna() & df[col2].isna())
        )

        target_comparison.loc[col1, col2] = (
            (~identical).sum()
        )

display(target_comparison)

In [ ]:
for col in sample_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

In [ ]:
feature_summary = pd.DataFrame({
    "non_missing": df[chs_features].notna().sum(),
    "missing": df[chs_features].isna().sum(),
    "missing_pct": df[chs_features].isna().mean() * 100,
    "mean": df[chs_features].mean(),
    "std": df[chs_features].std(),
    "min": df[chs_features].min(),
    "max": df[chs_features].max()
})

display(feature_summary.round(4))